# range_ladder — Optuna walk-forward (Phase A)

Tunes the **ladder structure** (rung counts, band placement, spacing
curvature, per-rung weight tilt) of the live `range_inventory_ladder`
controller with a 3-fold walk-forward on lake candles. Timing parameters are
FROZEN at live values (cooldown 3600 s; `executor_refresh_time` not modeled
— Phase B).

Papermill-parameterized: the **Parameters** cell below is tagged
`parameters`. Typical invocation:

```bash
papermill range_ladder_optuna_walkforward.ipynb out_XMR.ipynb \
    -p CONNECTOR nonkyc -p TRADING_PAIR XMR-USDT -p N_TRIALS 400
```

Supported connectors: `nonkyc`, `kraken` (fees/rules resolve per connector).
Live incumbents live at `configs/incumbents/<connector>__<pair>.yml`
(currently: nonkyc XMR/DASH/SUN/ZANO-USDT + kraken XMR-USD); a missing
file just skips the benchmark.

In [1]:
# Bootstrap: make pmm_lab importable, discover the subproject root, load .env.
import os
import sys
from pathlib import Path


def _find_subproject_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pmm_lab" / "__init__.py").exists():
            return base
    raise RuntimeError(f"pmm_dynamic subproject root not found above {Path.cwd()}")


SUBPROJECT_ROOT = _find_subproject_root()
if str(SUBPROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(SUBPROJECT_ROOT))

# .env discovery (MONGO_URI, OPTUNA_STORAGE) — walk up from the subproject root
_d = SUBPROJECT_ROOT
for _ in range(10):
    _env = _d / ".env"
    if _env.exists():
        for _line in _env.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _d = _d.parent

print(f"subproject root: {SUBPROJECT_ROOT}")

subproject root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


In [ ]:
# Parameters
CONNECTOR = "nonkyc"            # "nonkyc" | "kraken"
TRADING_PAIR = "XMR-USDT"       # nonkyc: XMR/DASH/SUN/ZANO-USDT; kraken: XMR-USDT, XMR-USD
INTERVAL = "1h"
N_TRIALS = 1000
N_STARTUP_TRIALS = 200
INCUMBENT_TRIAL = True          # benchmark + warm-start from a live YAML when present
FUND_USD = 1000.0               # deployed fund (quote units) — NOT tuned
QUOTE_FRAC = 0.5
N_JOBS = 1                      # >1 requires PostgreSQL OPTUNA_STORAGE
RUN_STRESS = True               # per-fold conservative re-score (informational)
STRESS_SPREAD_PCT = 0.0         # measured spread; stress slip = max(0.001, spread/2)
SEED = 12345
MIN_USABLE_DAYS = 150.0
MAX_GAP_PCT = 5.0
INCUMBENTS_DIR = "configs/incumbents"             # "" -> <subproject>/configs/incumbents
ARTIFACTS_DIR = "artifacts/range_ladder"

## 1. Environment + storage preflight

In [ ]:
import logging

import numpy as np
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print("MONGO_URI:", "SET" if MONGO_URI else "NOT SET")
print("OPTUNA_STORAGE:", "SET" if OPTUNA_STORAGE else "NOT SET (SQLite fallback)")

from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_pg = "postgresql" in _storage_url.lower()
if N_JOBS > 1 and not _is_pg:
    print(f"[preflight] N_JOBS={N_JOBS} with non-PostgreSQL storage -> forcing serial (N_JOBS=1)")
    N_JOBS = 1
print(f"[preflight] dispatch: {'process-parallel (PostgreSQL)' if N_JOBS > 1 else 'serial'}")

## 2. Data preflight (§5)

Audits lake coverage at 5m and the study interval, picks native bars when
present (else resamples 5m→1h), and ABORTS when usable history < 150 days or
the gap fraction exceeds 5% — no silent fold shrinking.

In [ ]:
from pmm_lab.config.defaults import INTERVAL_SECONDS
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.data.coverage import audit_pair, preflight_pair
from pmm_lab.data.hashing import hash_candles
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.optuna.objective_wrapper_range_ladder import plan_range_ladder_folds

loader = MongoCandleLoader()

print(f"=== Data preflight: {CONNECTOR} {TRADING_PAIR} ===")
for _iv in ("5m", INTERVAL):
    _r = audit_pair(CONNECTOR, TRADING_PAIR, _iv, loader=loader)
    print(f"  {_r['interval']:>4}: bars={_r['bars']:>8} days={_r['days']:8.1f} "
          f"gap_pct={_r['gap_pct']:6.2f}% max_gap_h={_r['max_gap_hours']:8.1f}")

candles, preflight_info = preflight_pair(
    CONNECTOR, TRADING_PAIR, interval=INTERVAL, loader=loader,
    min_usable_days=MIN_USABLE_DAYS, max_gap_pct=MAX_GAP_PCT,
)
print(f"source: {preflight_info['source']}, {len(candles)} bars")

bar_interval_seconds = INTERVAL_SECONDS[INTERVAL]
dataset_hash = hash_candles(candles)
reference_price = float(np.median(candles["close"]))
rules_db = load_exchange_rules()
pair_rules = resolve_pair_rules(rules_db, CONNECTOR, TRADING_PAIR)
maker_fee = pair_rules.fees.maker_fee
cooldown_bars = max(0, round(3600 / bar_interval_seconds))

train_days, test_days, step_days = plan_range_ladder_folds(len(candles), bar_interval_seconds)
print(f"fold plan: train={train_days:.1f}d, test={test_days:.1f}d x 3 folds (step={step_days:.1f}d)")
print(f"maker_fee={maker_fee} (dead-zone floor {4 * maker_fee:.4f}), cooldown_bars={cooldown_bars}")
print(f"dataset_hash={dataset_hash[:16]}..., reference_price={reference_price:.6g}")

## 3. Incumbent benchmark (§3.7)

When a live YAML exists at `configs/incumbents/<connector>__<pair>.yml`, the
LITERAL ladder is evaluated through the same fold machinery (benchmark row —
it cannot be an Optuna trial, raw prices are not in the search space), and a
least-squares generative approximation is enqueued to warm-start the study.
Missing incumbent → logged and skipped (e.g. Kraken).

In [ ]:
from pmm_lab.export.hb_yaml_range_ladder import (
    incumbent_yaml_path, load_range_ladder_incumbent,
)
from pmm_lab.features._numba_range_ladder import run_ladder_sim
from pmm_lab.objective.stress_range_ladder import run_range_ladder_stress
from pmm_lab.objective.walkforward import TimeSeriesCV
from pmm_lab.optuna.objective_wrapper_range_ladder import (
    ENDINV_GATE_PCT, winsorized_fold_objective,
)
from pmm_lab.strategies.range_ladder import RangeLadderConfig, compute_anchor
from pmm_lab.strategies.range_ladder_gen import (
    fit_generative_to_ladder, ladder_round_trip_error,
)

incumbent = None
incumbent_fit_params = None
incumbent_summary = None

if INCUMBENT_TRIAL:
    # A relative INCUMBENTS_DIR is anchored to the subproject root (the
    # kernel CWD is usually notebooks/range_ladder, NOT the repo).
    _dir = Path(INCUMBENTS_DIR) if INCUMBENTS_DIR else Path("configs/incumbents")
    if not _dir.is_absolute():
        _dir = SUBPROJECT_ROOT / _dir
    _path = incumbent_yaml_path(_dir, CONNECTOR, TRADING_PAIR)
    incumbent = load_range_ladder_incumbent(_path)
    if incumbent is None:
        print(f"no incumbent for ({CONNECTOR}, {TRADING_PAIR}) at {_path} — proceeding without a benchmark")

if incumbent is not None:
    lit_config = RangeLadderConfig(
        fund_quote=FUND_USD, quote_frac=QUOTE_FRAC, fee=maker_fee,
        cooldown_bars=cooldown_bars, stress_spread_pct=STRESS_SPREAD_PCT,
        literal_buy_prices=tuple(incumbent["buy_prices"]),
        literal_buy_weights=tuple(incumbent["buy_weights"]),
        literal_sell_prices=tuple(incumbent["sell_prices"]),
        literal_sell_weights=tuple(incumbent["sell_weights"]),
    )
    _rungs = lit_config.resolve_rungs(reference_price, pair_rules.price_tick)
    _cv = TimeSeriesCV(
        n_bars=len(candles), bar_interval_seconds=bar_interval_seconds,
        train_days=train_days, test_days=test_days, step_days=step_days,
        embargo_bars=0, macd_slow=3, natr_length=3,
    )
    _fold_rows = []
    for _fd in _cv.get_folds():
        _ts = candles[_fd.test_start_idx:_fd.test_end_idx]
        _t_days = len(_ts) * bar_interval_seconds / 86400.0
        _base = run_ladder_sim(
            _ts["open"], _ts["high"], _ts["low"], _ts["close"],
            _rungs.buys, _rungs.sells, _rungs.buy_weights, _rungs.sell_weights,
            fund=FUND_USD, quote_frac=QUOTE_FRAC, fee=maker_fee,
            cooldown_bars=cooldown_bars, bar_interval_seconds=bar_interval_seconds,
        )
        _cons = run_range_ladder_stress(_ts, lit_config, _rungs, bar_interval_seconds)
        _fold_rows.append(dict(
            fold=_fd.fold_index,
            score=_base["pnl_pct"] * 365.0 / _t_days,
            cons=_cons["pnl_pct"] * 365.0 / _t_days,
            endinv=_base["endinv_pct"],
            bf=int(sum(_base["buy_fills"])), sf=int(sum(_base["sell_fills"])),
        ))
    _scores = [r["score"] for r in _fold_rows]
    _obj, _ = winsorized_fold_objective(_scores)
    incumbent_summary = dict(objective=_obj, folds=_fold_rows)
    print("=== Incumbent benchmark (literal live ladder, same fold machinery) ===")
    for _r in _fold_rows:
        _gate = " GATE" if (_r["endinv"] > ENDINV_GATE_PCT or _r["bf"] < 1 or _r["sf"] < 1) else ""
        print(f"  fold {_r['fold']}: ann={_r['score']:8.1f}%  cons={_r['cons']:8.1f}%  "
              f"endinv={_r['endinv']:5.1f}%  fills={_r['bf']}b/{_r['sf']}s{_gate}")
    print(f"  incumbent winsorized objective: {_obj:.2f}")

    incumbent_fit_params = fit_generative_to_ladder(
        incumbent["buy_prices"], incumbent["buy_weights"],
        incumbent["sell_prices"], incumbent["sell_weights"],
    )
    _rt = ladder_round_trip_error(
        incumbent["buy_prices"], incumbent["buy_weights"],
        incumbent["sell_prices"], incumbent["sell_weights"], pair_rules.price_tick,
    )
    incumbent_fit_params.pop("anchor")
    print(f"generative approximation (round-trip err {_rt * 100:.2f}%): {incumbent_fit_params}")

## 4. Optuna study

HyperbandPruner acts on per-fold intermediate values; the first
`N_STARTUP_TRIALS` random trials are exempt (TPE + pruner startup).

In [ ]:
from pmm_lab.optuna.notebook_dispatch import optimize_study_for_notebook
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.study import create_study

study_name = f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_range_ladder_v1"
pruner = optuna.pruners.HyperbandPruner(min_resource=1, max_resource=3, reduction_factor=3)

# Create (or load) the study first so the incumbent approximation can be
# enqueued before optimization starts.
study = create_study(
    study_name=study_name, storage_url=_storage_url, seed=SEED,
    n_startup_trials=N_STARTUP_TRIALS, pruner=pruner,
)
if incumbent_fit_params is not None and len(study.trials) == 0:
    study.enqueue_trial(incumbent_fit_params)
    print("enqueued incumbent generative approximation as trial 0")

factory_kwargs = dict(
    candles=candles,
    pair_rules=pair_rules,
    bar_interval_seconds=bar_interval_seconds,
    dataset_hash=dataset_hash,
    reference_price=reference_price,
    strategy_name="range_ladder",
    train_days=train_days, test_days=test_days, step_days=step_days,
    run_stress=RUN_STRESS,
    fixed_quote=FUND_USD,
    objective_version=2,   # signature uniformity; ladder scores are annualized PnL%
)

if STRESS_SPREAD_PCT:
    # Thread the measured spread into every candidate config (serial only —
    # closures don't pickle for process-parallel dispatch).
    from dataclasses import replace as _replace
    from pmm_lab.optuna.canonicalizer_range_ladder import canonicalize_range_ladder_params

    def _canon_with_spread(raw, pr, ref, bar_interval_seconds=3600):
        bundle, reason = canonicalize_range_ladder_params(
            raw, pr, ref, bar_interval_seconds=bar_interval_seconds)
        if bundle is not None:
            bundle.strategy_config = _replace(
                bundle.strategy_config, stress_spread_pct=STRESS_SPREAD_PCT)
        return bundle, reason

    factory_kwargs["strategy_canonicalizer"] = _canon_with_spread
    N_JOBS = 1

study = optimize_study_for_notebook(
    study_name=study_name,
    storage_url=_storage_url,
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    objective_factory=create_objective,
    factory_kwargs=factory_kwargs,
    sampler_seed=SEED,
    n_startup_trials=N_STARTUP_TRIALS,
    pruner=pruner,
)

## 5. Results, export, report

In [ ]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
failed = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]
ranked = sorted(
    [t for t in completed if t.value is not None],
    key=lambda t: t.value, reverse=True,
)
print(f"trials: {len(study.trials)} total | {len(completed)} complete | "
      f"{len(pruned)} pruned | {len(failed)} failed")
if not ranked:
    raise RuntimeError(
        "NO COMPLETED TRIALS — every trial was pruned/rejected. "
        "Check the preflight audit and constraint floors before rerunning."
    )

print(f"\n=== Top 10 of {len(ranked)} completed trials ===")
_hdr = f"{'#':>4} {'objective':>10} {'pnl_med%':>9} {'endinv_med%':>11} {'cons_med':>9} {'viol':>4}"
print(_hdr)
for _t in ranked[:10]:
    _ua = _t.user_attrs
    print(f"{_t.number:>4} {_t.value:>10.2f} "
          f"{_ua.get('pnl_pct_median', float('nan')):>9.2f} "
          f"{_ua.get('endinv_pct_median', float('nan')):>11.1f} "
          f"{_ua.get('cons_score_median', float('nan')):>9.1f} "
          f"{_ua.get('gate_violations', '?'):>4}")
if incumbent_summary is not None:
    print(f"{'INC':>4} {incumbent_summary['objective']:>10.2f}   (literal live incumbent benchmark)")

best = ranked[0]
print(f"\nbest trial #{best.number}: objective={best.value:.2f}")
for _k, _v in sorted(best.params.items()):
    print(f"  {_k} = {_v}")

In [ ]:
from pmm_lab.export.hb_yaml_range_ladder import (
    RangeLadderExportParams, export_range_ladder_yaml,
)
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.optuna.canonicalizer_range_ladder import canonicalize_range_ladder_params

# Re-canonicalize the best trial with the study's fund settings
raw_best = dict(best.params)
raw_best["fund_quote"] = FUND_USD
raw_best["quote_frac"] = QUOTE_FRAC
raw_best["cooldown_time"] = 3600
bundle, reason = canonicalize_range_ladder_params(
    raw_best, pair_rules, reference_price, bar_interval_seconds=bar_interval_seconds)
if bundle is None:
    raise RuntimeError(f"best trial re-canonicalization failed: {reason}")
best_config = bundle.strategy_config

# Deploy-time anchor: latest median-3 close, loud warning on divergence
last_close = float(candles["close"][-1])
deploy_anchor = compute_anchor(candles["close"])
_div_pct = abs(deploy_anchor - last_close) / last_close * 100.0
if _div_pct > 2.0:
    print(f"!! WARNING: deploy anchor {deploy_anchor:.6g} diverges {_div_pct:.1f}% "
          f"from last close {last_close:.6g} -> trending, not ranging. "
          f"Re-check the market before deploying this export.")

out_dir = Path(ARTIFACTS_DIR) / CONNECTOR
yaml_path = out_dir / f"{TRADING_PAIR}_{INTERVAL}_screening_best.yml"
export_range_ladder_yaml(
    best_config, deploy_anchor, pair_rules,
    RangeLadderExportParams(connector_name=CONNECTOR, trading_pair=TRADING_PAIR),
    yaml_path, total_amount_quote=FUND_USD,
)
vr = validate_yaml_file(str(yaml_path), price_tick=pair_rules.price_tick)
print(f"exported: {yaml_path}  valid={vr.valid}")
for _w in vr.warnings:
    print(f"  warning: {_w}")
if not vr.valid:
    raise RuntimeError(f"export validation FAILED: {vr.errors}")

In [ ]:
# Markdown report (report_md section conventions, ladder-specific content)
import json
from datetime import datetime, timezone

_ua = best.user_attrs
_rungs_attr = _ua.get("last_fold_rungs", {})
_fold_detail = _ua.get("fold_detail", [])

_lines = [
    f"# range_ladder Phase A report — {CONNECTOR} {TRADING_PAIR} {INTERVAL}",
    "",
    f"- Generated: {datetime.now(timezone.utc).isoformat()}",
    f"- Study: `{study_name}`",
    f"- Dataset: {len(candles)} bars ({preflight_info['source']}), hash `{dataset_hash[:16]}...`",
    f"- Fold plan: train {train_days:.1f}d / test {test_days:.1f}d x 3 folds",
    f"- Fees: maker {maker_fee} ({CONNECTOR}), dead-zone floor {4 * maker_fee:.4f}",
    f"- Fund: {FUND_USD} quote, quote_frac {QUOTE_FRAC}",
    "",
    "## Trials",
    "",
    f"| total | complete | pruned | failed |",
    f"|---|---|---|---|",
    f"| {len(study.trials)} | {len(completed)} | {len(pruned)} | {len(failed)} |",
    "",
    "## Top 10",
    "",
    "| trial | objective | pnl_med % | endinv_med % | cons_med | gate viol |",
    "|---|---|---|---|---|---|",
]
for _t in ranked[:10]:
    _u = _t.user_attrs
    _lines.append(
        f"| {_t.number} | {_t.value:.2f} | {_u.get('pnl_pct_median', float('nan')):.2f} "
        f"| {_u.get('endinv_pct_median', float('nan')):.1f} "
        f"| {_u.get('cons_score_median', float('nan')):.1f} "
        f"| {_u.get('gate_violations', '?')} |")
if incumbent_summary is not None:
    _lines += [
        "",
        "## Incumbent benchmark (literal live ladder)",
        "",
        f"Winsorized objective: **{incumbent_summary['objective']:.2f}**",
        "",
        "| fold | ann % | cons ann % | endinv % | fills |",
        "|---|---|---|---|---|",
    ]
    for _r in incumbent_summary["folds"]:
        _lines.append(
            f"| {_r['fold']} | {_r['score']:.1f} | {_r['cons']:.1f} "
            f"| {_r['endinv']:.1f} | {_r['bf']}b/{_r['sf']}s |")
_lines += [
    "",
    "## Best trial",
    "",
    f"Trial #{best.number}, objective {best.value:.2f}",
    "",
    "```json",
    json.dumps(best.params, indent=2),
    "```",
    "",
    "### Per-fold detail",
    "",
    "```json",
    json.dumps(_fold_detail, indent=2, default=str),
    "```",
    "",
    "### Exported ladder (deploy anchor "
    f"{deploy_anchor:.6g}, divergence {_div_pct:.2f}% from last close)",
    "",
    "```json",
    json.dumps(_rungs_attr, indent=2),
    "```",
    "",
    f"Export: `{yaml_path}` (validated: {vr.valid})",
    "",
    "## Phase A caveats",
    "",
    "- No proceeds recycling; static per-rung quantities (Phase B).",
    "- `executor_refresh_time` not modeled (Phase B event-level sim).",
    "- Conservative (stress) scores are informational, not part of the objective.",
]
report_path = out_dir / f"{TRADING_PAIR}_{INTERVAL}_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(_lines), encoding="utf-8")
print(f"report: {report_path}")

## Notes

- **DASH-USDT / SUN-USDT** have no lake candles yet — the preflight aborts
  for them until the ingester backfills (MEXC proxy). XMR-USDT and ZANO-USDT
  (nonkyc) and XMR-USDT (kraken) have deep history.
- Kraken `XMR-USD` is thin in the lake; the preflight gate correctly rejects
  it until backfilled — do not weaken the gate.
- To benchmark an incumbent, drop the live controller YAML at
  `configs/incumbents/<connector>__<pair>.yml`.